# 05 - CNN Embedding-Distance Development Check

This notebook is a development check for the CNN image-embedding space. It compares same-class and different-class fabric pairs for the frozen DenseNet201 iteration and the fine-tuned DenseNet201 iteration. Microscopy images are embedded first, then averaged to one embedding vector per fabric before pairwise fabric distances are calculated.

The notebook includes cosine similarity and L2-normalised Euclidean distance to inspect the embedding space during development. The final thesis Results section does not rely on the Euclidean-distance output from this notebook; the reported image/spectra representation-similarity results are produced by notebooks `07` and `08`.

## Setup

The distance calculations are done directly with NumPy, without PGVector. Cosine similarity and L2-normalised Euclidean distance are used, so the visible distance checks focus on embedding direction rather than raw vector magnitude.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import random
from typing import Iterable

os.environ.setdefault("TORCH_HOME", str(Path.home() / ".cache" / "torch"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mdpp_v2_notebook_cache/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet201_Weights, densenet201

RANDOM_SEED = 42
BATCH_SIZE = 16
SPLITS = ["train", "val"]
IMAGE_MODALITY = "Stereo"
IMAGE_INSTRUMENT = "T"
SELECTED_SETTING = "Colour + mono + polarized grayscale"
TOP_K_VALUES = [1, 3, 5]

FROZEN_MODEL_LABEL = "iteration_1_frozen_densenet201"
FINETUNED_MODEL_LABEL = "iteration_2_finetuned_denseblock4"
MODEL_LABELS = [FROZEN_MODEL_LABEL, FINETUNED_MODEL_LABEL]
MODEL_DISPLAY_NAMES = {
    FROZEN_MODEL_LABEL: "Iteration 1: frozen DenseNet201",
    FINETUNED_MODEL_LABEL: "Iteration 2: fine-tuned denseblock4",
}

CLASS_NAMES = [
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Synthetic polyamide / nylon",
]
LABEL_TO_INDEX = {label: index for index, label in enumerate(CLASS_NAMES)}
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}

FEATURE_SOURCES = [
    {"feature_source": "C_gray", "formats": ("C",), "preprocessing": "L_to_RGB"},
    {"feature_source": "M_gray", "formats": ("M",), "preprocessing": "L_to_RGB"},
    {"feature_source": "P_gray", "formats": ("P",), "preprocessing": "L_to_RGB"},
]
SELECTED_FEATURE_SOURCES = tuple(source["feature_source"] for source in FEATURE_SOURCES)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))


def select_torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    print("CUDA is not available; using CPU for this inference notebook.")
    return torch.device("cpu")


DEVICE = select_torch_device()
PIN_MEMORY = DEVICE.type == "cuda"
DEVICE

In [ ]:
def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "mdpp_v2" / "data_splits").exists():
            return candidate / "mdpp_v2"
        if candidate.name == "mdpp_v2" and (candidate / "data_splits").exists():
            return candidate
    raise FileNotFoundError("Could not resolve mdpp_v2 root from current working directory.")

MDPP_ROOT = resolve_mdpp_root()
DATA_SPLITS_DIR = MDPP_ROOT / "data_splits"
ARTIFACTS_DIR = MDPP_ROOT / "artifacts"
FROZEN_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / "cnn_frozen_densenet201_linear_head.pt"
FINETUNED_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / "cnn_finetuned_densenet201_denseblock4_head.pt"

split_sample_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "samples.csv") for split in SPLITS}
split_manifest_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "measurements_manifest.csv") for split in SPLITS}

print(f"Using device: {DEVICE}")
print(f"Loaded splits: {', '.join(SPLITS)}")

## 1. Dataset and Embeddings

Each fabric can have multiple microscopy files. One DenseNet201 embedding is extracted per image for each CNN iteration, then those image embeddings are averaged to one fabric-level vector. Pairwise comparisons are made between those fabric vectors, so this development check is fabric-vs-fabric rather than replicate-vs-replicate.

In [ ]:
def eligible_samples_for_split(split: str) -> pd.DataFrame:
    samples = split_sample_tables[split].copy()
    eligible = samples[samples["cnn_label"].ne("EXCLUDED_CNN")].copy()
    eligible = eligible[["Sample ID", "classification_label", "cnn_label", "split"]]
    eligible = eligible.rename(columns={"Sample ID": "sample_id"})
    eligible["label_index"] = eligible["classification_label"].map(LABEL_TO_INDEX)
    if eligible["label_index"].isna().any():
        missing = sorted(eligible.loc[eligible["label_index"].isna(), "classification_label"].dropna().unique())
        raise ValueError(f"Unexpected retained CNN labels: {missing}")
    return eligible


def build_source_table(split: str, feature_source: str, formats: Iterable[str], preprocessing: str) -> pd.DataFrame:
    manifest = split_manifest_tables[split].copy()
    eligible = eligible_samples_for_split(split)
    mask = (
        manifest["modality_group"].eq("image")
        & manifest["modality"].eq(IMAGE_MODALITY)
        & manifest["instrument"].eq(IMAGE_INSTRUMENT)
        & manifest["data_format"].isin(tuple(formats))
    )
    table = manifest.loc[mask].merge(eligible, on="sample_id", how="inner", suffixes=("", "_sample"))
    table["path"] = table["final_path"].map(Path)
    table["feature_source"] = feature_source
    table["preprocessing"] = preprocessing
    table["setting"] = SELECTED_SETTING
    table["source_split"] = split
    table["image_id"] = table["source_split"] + ":" + table["feature_source"] + ":" + table["filename"]
    if table["cnn_label"].eq("EXCLUDED_CNN").any():
        raise ValueError("EXCLUDED_CNN rows entered the classifier table.")
    missing_files = [path for path in table["path"] if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing image files, first examples: {missing_files[:3]}")
    return table.sort_values(["sample_id", "feature_source", "replicate", "filename"]).reset_index(drop=True)


def fabric_label_table(table: pd.DataFrame) -> pd.DataFrame:
    return (
        table[["sample_id", "classification_label", "label_index", "source_split"]]
        .drop_duplicates("sample_id")
        .sort_values("sample_id")
        .reset_index(drop=True)
    )

source_tables = {}
for source_spec in FEATURE_SOURCES:
    feature_source = source_spec["feature_source"]
    source_tables[feature_source] = {}
    for split in SPLITS:
        source_tables[feature_source][split] = build_source_table(
            split=split,
            feature_source=feature_source,
            formats=source_spec["formats"],
            preprocessing=source_spec["preprocessing"],
        )

image_tables = {}
for split in SPLITS:
    image_tables[split] = pd.concat(
        [source_tables[source][split] for source in SELECTED_FEATURE_SOURCES],
        ignore_index=True,
    ).sort_values(["sample_id", "feature_source", "replicate", "filename"]).reset_index(drop=True)

development_table = pd.concat([image_tables[split] for split in SPLITS], ignore_index=True)
development_fabrics = fabric_label_table(development_table)

def class_count_string(frame: pd.DataFrame) -> str:
    counts = frame["classification_label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    return "; ".join(f"{label}: {int(counts.loc[label])}" for label in CLASS_NAMES)

summary_rows = []
for split in SPLITS:
    split_table = image_tables[split]
    split_fabrics = fabric_label_table(split_table)
    source_counts = split_table["data_format"].value_counts().reindex(["C", "M", "P"], fill_value=0)
    summary_rows.append({
        "scope": split,
        "fabric_samples": split_table["sample_id"].nunique(),
        "image_files": len(split_table),
        "C_files": int(source_counts.loc["C"]),
        "M_files": int(source_counts.loc["M"]),
        "P_files": int(source_counts.loc["P"]),
        "class_counts": class_count_string(split_fabrics),
    })
source_counts = development_table["data_format"].value_counts().reindex(["C", "M", "P"], fill_value=0)
summary_rows.append({
    "scope": "train + validation",
    "fabric_samples": development_table["sample_id"].nunique(),
    "image_files": len(development_table),
    "C_files": int(source_counts.loc["C"]),
    "M_files": int(source_counts.loc["M"]),
    "P_files": int(source_counts.loc["P"]),
    "class_counts": class_count_string(development_fabrics),
})

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary


In [ ]:
weights = DenseNet201_Weights.DEFAULT
weights_transform = weights.transforms()

class MicroscopyImageDataset(Dataset):
    def __init__(self, table: pd.DataFrame):
        self.table = table.reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int):
        row = self.table.iloc[idx]
        with Image.open(row["path"]) as image:
            if row["preprocessing"] == "L_to_RGB":
                image = image.convert("L").convert("RGB")
            elif row["preprocessing"] == "RGB":
                image = image.convert("RGB")
            else:
                raise ValueError(f"Unknown preprocessing mode: {row['preprocessing']}")
        return weights_transform(image), int(row["label_index"]), row["sample_id"], row["filename"], row["feature_source"], row["source_split"]


def make_loader(table: pd.DataFrame) -> DataLoader:
    return DataLoader(MicroscopyImageDataset(table), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)


class DenseNetEmbeddingExtractor(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.features = model.features

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images)
        features = F.relu(features, inplace=False)
        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        return torch.flatten(pooled, 1)


def load_checkpoint(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}")
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError(f"Checkpoint class names do not match CLASS_NAMES: {path}")
    return checkpoint


def make_frozen_embedding_model() -> tuple[nn.Module, dict]:
    checkpoint = load_checkpoint(FROZEN_CNN_CHECKPOINT_PATH)
    model = densenet201(weights=weights)
    feature_dim = model.classifier.in_features
    extractor = DenseNetEmbeddingExtractor(model).to(DEVICE).eval()
    for parameter in extractor.parameters():
        parameter.requires_grad = False
    return extractor, {
        "model": FROZEN_MODEL_LABEL,
        "checkpoint": FROZEN_CNN_CHECKPOINT_PATH.name,
        "checkpoint_best_epoch": checkpoint.get("best_epoch"),
        "feature_dim": feature_dim,
        "trainable_parameters": sum(parameter.numel() for parameter in extractor.parameters() if parameter.requires_grad),
    }


def make_finetuned_embedding_model() -> tuple[nn.Module, dict]:
    checkpoint = load_checkpoint(FINETUNED_CNN_CHECKPOINT_PATH)
    model = densenet201(weights=weights)
    feature_dim = model.classifier.in_features
    model.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))
    mapped_tail_state = {}
    for key, value in checkpoint["tail_state_dict"].items():
        if key.startswith(("denseblock4.", "norm5.")):
            mapped_tail_state[f"features.{key}"] = value
        elif key.startswith("classifier."):
            mapped_tail_state[key] = value
        else:
            raise ValueError(f"Unexpected fine-tuned checkpoint key: {key}")
    load_result = model.load_state_dict(mapped_tail_state, strict=False)
    if load_result.unexpected_keys:
        raise ValueError(f"Unexpected keys while loading fine-tuned model: {load_result.unexpected_keys}")
    extractor = DenseNetEmbeddingExtractor(model).to(DEVICE).eval()
    for parameter in extractor.parameters():
        parameter.requires_grad = False
    return extractor, {
        "model": FINETUNED_MODEL_LABEL,
        "checkpoint": FINETUNED_CNN_CHECKPOINT_PATH.name,
        "checkpoint_best_epoch": checkpoint.get("best_epoch"),
        "feature_dim": feature_dim,
        "trainable_parameters": sum(parameter.numel() for parameter in extractor.parameters() if parameter.requires_grad),
    }


@torch.no_grad()
def extract_features(table: pd.DataFrame, model: nn.Module, model_label: str) -> tuple[pd.DataFrame, np.ndarray]:
    loader = make_loader(table)
    rows = []
    feature_batches = []
    for images, labels, sample_ids, filenames, feature_sources, source_splits in loader:
        features = model(images.to(DEVICE, non_blocking=PIN_MEMORY)).detach().cpu().numpy()
        feature_batches.append(features)
        for label, sample_id, filename, feature_source, source_split in zip(labels.numpy(), sample_ids, filenames, feature_sources, source_splits):
            rows.append({
                "model": model_label,
                "sample_id": sample_id,
                "filename": filename,
                "feature_source": feature_source,
                "label_index": int(label),
                "classification_label": INDEX_TO_LABEL[int(label)],
                "source_split": source_split,
            })
    return pd.DataFrame(rows), np.vstack(feature_batches).astype("float32")


embedding_models = []
model_metadata_rows = []
for model_label, factory in [
    (FROZEN_MODEL_LABEL, make_frozen_embedding_model),
    (FINETUNED_MODEL_LABEL, make_finetuned_embedding_model),
]:
    model, metadata = factory()
    embedding_models.append((model_label, model))
    model_metadata_rows.append(metadata)

model_metadata = pd.DataFrame(model_metadata_rows)
model_feature_frames = []
model_feature_matrices = {}
for model_label, model in embedding_models:
    feature_frame, feature_matrix = extract_features(development_table, model, model_label)
    model_feature_frames.append(feature_frame)
    model_feature_matrices[model_label] = feature_matrix
    print(f"{model_label}: extracted {len(feature_frame)} image embeddings for {feature_frame['sample_id'].nunique()} fabrics.")

image_embedding_frame = pd.concat(model_feature_frames, ignore_index=True)
embedding_extraction_summary = model_metadata.merge(
    image_embedding_frame.groupby("model", as_index=False).agg(
        image_embeddings=("sample_id", "size"),
        fabric_samples=("sample_id", "nunique"),
    ),
    on="model",
    how="left",
)
embedding_extraction_summary

In [ ]:
def average_embeddings_by_fabric(frame: pd.DataFrame, matrix: np.ndarray) -> tuple[pd.DataFrame, np.ndarray]:
    rows = []
    vectors = []
    for sample_id, group in frame.groupby("sample_id", sort=True):
        indices = group.index.to_numpy()
        labels = group["classification_label"].unique()
        source_splits = group["source_split"].unique()
        models = group["model"].unique()
        if len(labels) != 1:
            raise ValueError(f"Multiple labels found for {sample_id}: {labels}")
        if len(source_splits) != 1:
            raise ValueError(f"Multiple source splits found for {sample_id}: {source_splits}")
        if len(models) != 1:
            raise ValueError(f"Multiple model labels found for {sample_id}: {models}")
        vector = matrix[indices].mean(axis=0).astype("float32")
        rows.append({
            "model": models[0],
            "sample_id": sample_id,
            "classification_label": labels[0],
            "source_split": source_splits[0],
            "image_count": len(indices),
            "feature_sources": ", ".join(sorted(group["feature_source"].unique())),
        })
        vectors.append(vector)
    return pd.DataFrame(rows), np.vstack(vectors).astype("float32")


def l2_normalise(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Found zero-norm fabric embedding.")
    return (matrix / norms).astype("float32")


fabric_embedding_frames = []
fabric_matrices = {}
fabric_normalised_matrices = {}
for model_label in MODEL_LABELS:
    frame = image_embedding_frame[image_embedding_frame["model"].eq(model_label)].reset_index(drop=True)
    matrix = model_feature_matrices[model_label]
    fabric_frame, fabric_matrix = average_embeddings_by_fabric(frame, matrix)
    fabric_embedding_frames.append(fabric_frame)
    fabric_matrices[model_label] = fabric_matrix
    fabric_normalised_matrices[model_label] = l2_normalise(fabric_matrix)

fabric_embeddings = pd.concat(fabric_embedding_frames, ignore_index=True)
fabric_embedding_summary = fabric_embeddings.groupby("model", as_index=False).agg(
    fabric_embeddings=("sample_id", "size"),
    image_files=("image_count", "sum"),
)
print(f"Averaged embeddings for {len(MODEL_LABELS)} CNN iterations.")
fabric_embedding_summary

## 2. Embedding-Distance Results

The same-class group contains pairs where both fabrics have the same `classification_label`. The different-class group contains pairs with different labels. For cosine similarity, higher means closer. For L2-normalised Euclidean distance, lower means closer. The table is used as development evidence for how the CNN embedding space behaves; it is not the final image/spectra representation-similarity result reported in the thesis.

In [ ]:
def pairwise_distances_for_model(model_label: str) -> tuple[pd.DataFrame, float]:
    model_fabrics = fabric_embeddings[fabric_embeddings["model"].eq(model_label)].reset_index(drop=True)
    sample_ids = model_fabrics["sample_id"].to_numpy()
    labels = model_fabrics["classification_label"].to_numpy()
    splits = model_fabrics["source_split"].to_numpy()
    matrix = fabric_normalised_matrices[model_label]
    row_indices, col_indices = np.triu_indices(len(model_fabrics), k=1)

    cosine_similarity_values = np.sum(matrix[row_indices] * matrix[col_indices], axis=1)
    cosine_distance_values = 1.0 - cosine_similarity_values
    normalised_euclidean_values = np.linalg.norm(matrix[row_indices] - matrix[col_indices], axis=1)
    expected_normalised_euclidean_values = np.sqrt(np.maximum(0.0, 2.0 * cosine_distance_values))
    relation_error = float(np.max(np.abs(normalised_euclidean_values - expected_normalised_euclidean_values)))

    frame = pd.DataFrame({
        "model": model_label,
        "sample_id_a": sample_ids[row_indices],
        "label_a": labels[row_indices],
        "split_a": splits[row_indices],
        "sample_id_b": sample_ids[col_indices],
        "label_b": labels[col_indices],
        "split_b": splits[col_indices],
        "same_class": labels[row_indices] == labels[col_indices],
        "cosine_similarity": cosine_similarity_values,
        "cosine_distance": cosine_distance_values,
        "normalised_euclidean_distance": normalised_euclidean_values,
        "expected_normalised_euclidean_distance": expected_normalised_euclidean_values,
    })
    frame["pair_type"] = np.where(frame["same_class"], "same class", "different class")
    frame["class_pair"] = frame.apply(lambda row: " / ".join(sorted([row["label_a"], row["label_b"]])), axis=1)
    return frame, relation_error


pairwise_frames = []
relation_error_rows = []
for model_label in MODEL_LABELS:
    frame, relation_error = pairwise_distances_for_model(model_label)
    pairwise_frames.append(frame)
    relation_error_rows.append({"model": model_label, "max_abs_error": relation_error})

pairwise_distances = pd.concat(pairwise_frames, ignore_index=True)
normalised_euclidean_relation_errors = pd.DataFrame(relation_error_rows)
normalised_euclidean_relation_error = float(normalised_euclidean_relation_errors["max_abs_error"].max())


def grouped_stat(frame: pd.DataFrame, metric: str, pair_type: str, stat: str) -> float:
    values = frame.loc[frame["pair_type"].eq(pair_type), metric]
    if stat == "mean":
        return float(values.mean())
    if stat == "median":
        return float(values.median())
    raise ValueError(stat)


def scope_pair_frame(scope: str, model_label: str | None = None) -> pd.DataFrame:
    frame = pairwise_distances if model_label is None else pairwise_distances[pairwise_distances["model"].eq(model_label)]
    if scope == "train":
        return frame[frame["split_a"].eq("train") & frame["split_b"].eq("train")].copy()
    if scope == "validation":
        return frame[frame["split_a"].eq("val") & frame["split_b"].eq("val")].copy()
    if scope == "train + validation":
        return frame.copy()
    raise ValueError(scope)


def metric_summary_row(frame: pd.DataFrame, model_label: str, scope: str, metric_label: str, metric_column: str, closer_direction: str) -> dict:
    same_mean = grouped_stat(frame, metric_column, "same class", "mean")
    different_mean = grouped_stat(frame, metric_column, "different class", "mean")
    same_median = grouped_stat(frame, metric_column, "same class", "median")
    different_median = grouped_stat(frame, metric_column, "different class", "median")
    if closer_direction == "higher is closer":
        mean_gap = same_mean - different_mean
        median_gap = same_median - different_median
        passes_direction_check = same_mean > different_mean
    elif closer_direction == "lower is closer":
        mean_gap = different_mean - same_mean
        median_gap = different_median - same_median
        passes_direction_check = different_mean > same_mean
    else:
        raise ValueError(closer_direction)
    return {
        "model": model_label,
        "scope": scope,
        "metric": metric_label,
        "closer_direction": closer_direction,
        "fabric_samples": int(len(set(frame["sample_id_a"]) | set(frame["sample_id_b"]))),
        "same_class_pairs": int(frame["same_class"].sum()),
        "different_class_pairs": int((~frame["same_class"]).sum()),
        "same_mean": same_mean,
        "different_mean": different_mean,
        "mean_gap": mean_gap,
        "same_median": same_median,
        "different_median": different_median,
        "median_gap": median_gap,
        "passes_direction_check": passes_direction_check,
    }


def similarity_rows_for_scope(model_label: str, scope: str) -> list[dict]:
    frame = scope_pair_frame(scope, model_label)
    return [
        metric_summary_row(frame, model_label, scope, "cosine similarity", "cosine_similarity", "higher is closer"),
        metric_summary_row(frame, model_label, scope, "normalised Euclidean distance", "normalised_euclidean_distance", "lower is closer"),
    ]


scope_order = {"train": 0, "validation": 1, "train + validation": 2}
metric_order = {"cosine similarity": 0, "normalised Euclidean distance": 1}
embedding_similarity_result = pd.DataFrame(
    row
    for model_label in MODEL_LABELS
    for scope in ["train", "validation", "train + validation"]
    for row in similarity_rows_for_scope(model_label, scope)
)
embedding_similarity_result = (
    embedding_similarity_result
    .assign(scope_order=lambda frame: frame["scope"].map(scope_order), metric_order=lambda frame: frame["metric"].map(metric_order))
    .sort_values(["model", "scope_order", "metric_order"])
    .drop(columns=["scope_order", "metric_order"])
    .reset_index(drop=True)
)

same_pair_count = int(scope_pair_frame("train + validation", FROZEN_MODEL_LABEL)["same_class"].sum())
different_pair_count = int((~scope_pair_frame("train + validation", FROZEN_MODEL_LABEL)["same_class"]).sum())
embedding_similarity_result

## 3. Distance Distributions

These plots show whether same-class and different-class pairs occupy different parts of the embedding space. Some overlap is expected because the labels are coarse dominant-fibre labels and visual texture can overlap across fibre classes.

In [ ]:
plot_colours = {"same class": "#2563eb", "different class": "#dc2626"}
plot_order = ["same class", "different class"]
plot_labels = ["same class", "different class"]
plot_colors = [plot_colours[label] for label in plot_order]
fig, axes = plt.subplots(len(MODEL_LABELS), 2, figsize=(13, 7.8), constrained_layout=True)

for row_index, model_label in enumerate(MODEL_LABELS):
    frame = pairwise_distances[pairwise_distances["model"].eq(model_label)]
    cosine_values = [
        frame.loc[frame["pair_type"].eq(pair_type), "cosine_similarity"]
        for pair_type in plot_order
    ]
    euclidean_values = [
        frame.loc[frame["pair_type"].eq(pair_type), "normalised_euclidean_distance"]
        for pair_type in plot_order
    ]
    axes[row_index, 0].hist(cosine_values, bins=35, label=plot_labels, color=plot_colors, alpha=1.0, histtype="bar")
    axes[row_index, 1].hist(euclidean_values, bins=35, label=plot_labels, color=plot_colors, alpha=1.0, histtype="bar")
    axes[row_index, 0].set_title(f"{MODEL_DISPLAY_NAMES[model_label]}: cosine similarity")
    axes[row_index, 0].set_xlabel("Cosine similarity")
    axes[row_index, 0].set_ylabel("Fabric pairs")
    axes[row_index, 1].set_title(f"{MODEL_DISPLAY_NAMES[model_label]}: normalised Euclidean distance")
    axes[row_index, 1].set_xlabel("Normalised Euclidean distance")
    axes[row_index, 1].set_ylabel("Fabric pairs")
    for ax in axes[row_index]:
        ax.legend()
        ax.grid(axis="y", alpha=0.2)
plt.show()

## 4. Class-Pair Medians

The heatmaps summarise median pairwise distance per class combination. The diagonal shows same-class comparisons.

In [ ]:
def class_pair_matrix(frame: pd.DataFrame, metric: str) -> pd.DataFrame:
    matrix = pd.DataFrame(np.nan, index=CLASS_NAMES, columns=CLASS_NAMES, dtype=float)
    for i, class_a in enumerate(CLASS_NAMES):
        for j, class_b in enumerate(CLASS_NAMES):
            if i <= j:
                mask = (
                    ((frame["label_a"].eq(class_a)) & (frame["label_b"].eq(class_b)))
                    | ((frame["label_a"].eq(class_b)) & (frame["label_b"].eq(class_a)))
                )
                if mask.any():
                    value = float(frame.loc[mask, metric].median())
                    matrix.loc[class_a, class_b] = value
                    matrix.loc[class_b, class_a] = value
    return matrix

fig, axes = plt.subplots(len(MODEL_LABELS), 2, figsize=(16, 10), constrained_layout=True)
for row_index, model_label in enumerate(MODEL_LABELS):
    frame = pairwise_distances[pairwise_distances["model"].eq(model_label)]
    matrices = [
        (class_pair_matrix(frame, "cosine_similarity"), "Median cosine similarity", "Blues"),
        (class_pair_matrix(frame, "normalised_euclidean_distance"), "Median normalised Euclidean distance", "Oranges"),
    ]
    for col_index, (matrix, title, cmap) in enumerate(matrices):
        ax = axes[row_index, col_index]
        image = ax.imshow(matrix.to_numpy(), cmap=cmap)
        ax.set_title(f"{MODEL_DISPLAY_NAMES[model_label]}\n{title}")
        ax.set_xticks(range(len(CLASS_NAMES)))
        ax.set_yticks(range(len(CLASS_NAMES)))
        ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
        ax.set_yticklabels(CLASS_NAMES)
        for row in range(len(CLASS_NAMES)):
            for col in range(len(CLASS_NAMES)):
                value = matrix.iloc[row, col]
                if np.isfinite(value):
                    ax.text(col, row, f"{value:.3f}", ha="center", va="center", fontsize=8)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## 5. Hard Negatives and Nearest Neighbours

Hard negatives are different-class pairs that are still close in the embedding space. They show where a representation places visually similar but differently labelled fabrics near each other.

In [ ]:
hard_negative_columns = [
    "model",
    "sample_id_a",
    "label_a",
    "sample_id_b",
    "label_b",
    "cosine_similarity",
    "normalised_euclidean_distance",
]

hard_negative_frames = []
for model_label in MODEL_LABELS:
    frame = pairwise_distances[pairwise_distances["model"].eq(model_label) & ~pairwise_distances["same_class"]]
    cosine_hard_negatives = (
        frame
        .sort_values(["cosine_similarity", "normalised_euclidean_distance"], ascending=[False, True])
        .head(5)[hard_negative_columns]
        .assign(hard_negative_type="highest cosine similarity")
    )
    euclidean_hard_negatives = (
        frame
        .sort_values(["normalised_euclidean_distance", "cosine_similarity"], ascending=[True, False])
        .head(5)[hard_negative_columns]
        .assign(hard_negative_type="lowest normalised Euclidean distance")
    )
    hard_negative_frames.extend([cosine_hard_negatives, euclidean_hard_negatives])

hard_negative_examples = pd.concat(hard_negative_frames, ignore_index=True)
hard_negative_examples = hard_negative_examples[["model", "hard_negative_type", *[column for column in hard_negative_columns if column != "model"]]]
hard_negative_examples

In [ ]:
def nearest_neighbour_details(model_label: str, metric_name: str, matrix: np.ndarray, higher_is_closer: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_fabrics = fabric_embeddings[fabric_embeddings["model"].eq(model_label)].reset_index(drop=True)
    sample_ids = model_fabrics["sample_id"].to_numpy()
    labels = model_fabrics["classification_label"].to_numpy()
    rows = []
    topk_rows = []
    for query_index, sample_id in enumerate(sample_ids):
        order = np.argsort(-matrix[query_index]) if higher_is_closer else np.argsort(matrix[query_index])
        order = order[order != query_index]
        nearest_index = int(order[0])
        rows.append({
            "model": model_label,
            "metric": metric_name,
            "sample_id": sample_id,
            "classification_label": labels[query_index],
            "nearest_sample_id": sample_ids[nearest_index],
            "nearest_label": labels[nearest_index],
            "nearest_same_class": bool(labels[query_index] == labels[nearest_index]),
            "nearest_value": float(matrix[query_index, nearest_index]),
        })
        for k in TOP_K_VALUES:
            top_indices = order[:k]
            topk_rows.append({
                "model": model_label,
                "metric": metric_name,
                "sample_id": sample_id,
                "k": k,
                "same_class_share": float(np.mean(labels[top_indices] == labels[query_index])),
                "has_any_same_class": bool(np.any(labels[top_indices] == labels[query_index])),
            })
    return pd.DataFrame(rows), pd.DataFrame(topk_rows)

nearest_frames = []
topk_frames = []
for model_label in MODEL_LABELS:
    normalised_matrix = fabric_normalised_matrices[model_label]
    cosine_matrix = normalised_matrix @ normalised_matrix.T
    euclidean_matrix = np.linalg.norm(normalised_matrix[:, None, :] - normalised_matrix[None, :, :], axis=2)
    np.fill_diagonal(cosine_matrix, -np.inf)
    np.fill_diagonal(euclidean_matrix, np.inf)
    cosine_nearest, cosine_topk = nearest_neighbour_details(model_label, "cosine similarity", cosine_matrix, higher_is_closer=True)
    euclidean_nearest, euclidean_topk = nearest_neighbour_details(model_label, "normalised Euclidean distance", euclidean_matrix, higher_is_closer=False)
    nearest_frames.extend([cosine_nearest, euclidean_nearest])
    topk_frames.extend([cosine_topk, euclidean_topk])

nearest_details = pd.concat(nearest_frames, ignore_index=True)
topk_details = pd.concat(topk_frames, ignore_index=True)

nearest_summary = nearest_details.groupby(["model", "metric"], as_index=False).agg(
    fabrics=("sample_id", "count"),
    top1_same_class_rate=("nearest_same_class", "mean"),
)
topk_summary = topk_details.groupby(["model", "metric", "k"], as_index=False).agg(
    mean_same_class_share=("same_class_share", "mean"),
    any_same_class_rate=("has_any_same_class", "mean"),
)
nearest_summary = nearest_summary.merge(
    topk_summary.pivot(index=["model", "metric"], columns="k", values="mean_same_class_share").add_prefix("topk_mean_same_class_share_").reset_index(),
    on=["model", "metric"],
    how="left",
)
nearest_summary

## 6. Development Values

Key values from the embedding-distance development check.

In [ ]:
result_values = embedding_similarity_result[
    embedding_similarity_result["scope"].isin(["validation", "train + validation"])
].merge(
    nearest_summary[["model", "metric", "top1_same_class_rate"]],
    on=["model", "metric"],
    how="left",
)
result_values = result_values[[
    "model",
    "scope",
    "metric",
    "same_mean",
    "different_mean",
    "mean_gap",
    "same_median",
    "different_median",
    "median_gap",
    "passes_direction_check",
    "top1_same_class_rate",
]]
result_values

## 7. Interpretation

In [ ]:
combined_results = embedding_similarity_result[embedding_similarity_result["scope"].eq("train + validation")]
for model_label in MODEL_LABELS:
    model_results = combined_results[combined_results["model"].eq(model_label)]
    cosine_row = model_results[model_results["metric"].eq("cosine similarity")].iloc[0]
    euclidean_row = model_results[model_results["metric"].eq("normalised Euclidean distance")].iloc[0]
    print(MODEL_DISPLAY_NAMES[model_label])
    print(
        f"  Cosine similarity: same-class pairs are higher by {cosine_row['mean_gap']:.4f} on average "
        f"and {cosine_row['median_gap']:.4f} at the median."
    )
    print(
        f"  Normalised Euclidean distance: different-class pairs are farther by {euclidean_row['mean_gap']:.4f} on average "
        f"and {euclidean_row['median_gap']:.4f} at the median."
    )
print("The embedding-distance table shows whether the same pattern appears within train and validation separately.")